# Actor Clustering - Node2Vec + KMeans

This notebook implements the actor-level clustering pipeline for one model using the shared preprocessing and graph construction logic.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.clustering.node2vec import run_node2vec_kmeans
from src.preprocessing.pipeline import prepare_common_actor_data
from src.utils.config import RESULTS_DIR, SEED
from src.utils.io_utils import save_json, save_results_csv

MAX_NODES = 2000
N_CLUSTERS = 8
common = prepare_common_actor_data(max_nodes=MAX_NODES)
assignments, embedding, metrics = run_node2vec_kmeans(common["graph"], n_clusters=N_CLUSTERS, seed=SEED)
result_dir = RESULTS_DIR / "node2vec_kmeans"
save_results_csv(assignments, result_dir / "cluster_assignments.csv")
save_json(metrics, result_dir / "metrics.json")
import numpy as np
np.save(result_dir / "embeddings.npy", embedding)
print(metrics)

{'method': 'node2vec_kmeans', 'n_nodes': 2000, 'n_edges': 1031, 'n_clusters': 8, 'embedding_dimensions': 32, 'runtime_seconds': 1.0877070010028547, 'seed': 42}


In [2]:
from src.clustering.evaluation import evaluate_clusters
from src.clustering.visualization import save_cluster_figures, save_graph_partition_figure

labels = common["actor_df"].set_index("actor_id")["class"].reindex(assignments["actor_id"])
metrics.update(evaluate_clusters(assignments, embedding, labels))
metrics["research_questions"] = [
    "Does a learned random-walk embedding capture actor similarity beyond direct AddrAddr communities?",
    "Does Node2Vec improve label-aligned separation without using labels during training?",
]
save_json(metrics, result_dir / "metrics.json")
save_cluster_figures(assignments, result_dir, embedding, title="Node2Vec actor communities")
print("Research interpretation metrics:", {key: metrics[key] for key in ("silhouette", "ari", "nmi", "ami")})

Research interpretation metrics: {'silhouette': 0.6297745704650879, 'ari': -0.015304495318496935, 'nmi': 0.004022041963175511, 'ami': -0.0009828570365961317}
